# Single-cell data preparation
Run locally from the repository root, using the analysis environment and separately obtained source data. Set SCPCA_H5AD and SCPCA_METADATA before launching Jupyter. No model provider receives data in this notebook.

Run sections 1–3 for training/validation data. Review the generated technical QC plots before section 4. Section 5 accesses reserved donors using the same reviewed recipe. Section 6 is run **later**, after expert test review and matched validation, and before any test model inference. Existing output folders are intentionally protected from overwrite.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'src').is_dir(), 'Launch from the repository or notebooks folder.'
sys.path.insert(0, str(ROOT/'src'))
print(ROOT)

## 1. Metadata-only donor split
This is the actual scientific donor-selection method. It does not read expression values or reference cell-type labels.

In [ ]:
"""Freeze a metadata-only donor split; do not read expression or annotation labels."""
import csv
import os
import gzip
import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import h5py
import numpy as np
import pandas as pd


SOURCE = Path(os.environ.get('SCPCA_DATA_DIR', ROOT / 'data/source'))
META = Path(os.environ.get('SCPCA_METADATA', str(SOURCE / 'SCPCP000004_metadata_2026-09-13/metadata.tsv')))
H5AD = Path(os.environ.get('SCPCA_H5AD', str(SOURCE / 'SCPCP000004_ann-data_2026-09-13/SCPCP000004_single-cell_merged/SCPCP000004_merged_rna.h5ad')))
OUT = ROOT / 'outputs/patient-benchmark-v1'
SALT = 'GTC2027-SCPCP000004-patient-split-v1-20261031'
QUARANTINE = {'SJNBL046', 'SJNBL108'}
PARTITION_COUNTS = {'train': 12, 'development': 4, 'test': 4}


def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def vector(group, name):
    node = group[name]
    if isinstance(node, h5py.Dataset):
        return node.asstr()[()] if node.dtype.kind in 'OSU' else node[()]
    if 'categories' in node and 'codes' in node:
        categories = node['categories'].asstr()[()]
        codes = node['codes'][()]
        return np.asarray([categories[c] if c >= 0 else None for c in codes], dtype=object)
    raise ValueError(f'Unsupported encoding in {name}')


def write_csv(name, frame):
    frame.to_csv(OUT / name, index=False, lineterminator='\n')


def main():
    OUT.mkdir(parents=True, exist_ok=False)
    metadata = pd.read_csv(META, sep='\t', dtype=str, keep_default_na=False)
    assert metadata.scpca_library_id.is_unique
    assert metadata.groupby('scpca_sample_id').participant_id.nunique().max() == 1
    # No expression, reference annotation, embedding or model answer is read.
    read_fields = ['_index', 'library_id', 'sample_id', 'participant_id', 'is_xenograft',
                   'is_cell_line', 'miQC_pass', 'scDblFinder_class', 'scpca_filter', 'detected', 'sum']
    with h5py.File(H5AD, 'r') as f:
        obs = pd.DataFrame({name: vector(f['obs'], name) for name in read_fields})
        shape = tuple(int(x) for x in f['X'].attrs['shape'])
        x_name = f['uns/X_name'].asstr()[()]
    assert len(obs) == shape[0] and obs['_index'].is_unique
    obs.insert(0, 'source_row', np.arange(len(obs)))
    meta_index = metadata.set_index('scpca_library_id')
    assert set(obs.library_id) == set(meta_index.index)
    for field, column in [('sample_id', 'scpca_sample_id'), ('participant_id', 'participant_id')]:
        expected = obs.library_id.map(meta_index[column])
        assert (obs[field].to_numpy() == expected.to_numpy()).all(), f'Metadata mismatch: {field}'
    for field in ['is_xenograft', 'is_cell_line']:
        expected = obs.library_id.map(meta_index[field]).str.lower().eq('true')
        assert np.array_equal(obs[field].astype(bool), expected), f'Metadata mismatch: {field}'

    reason = pd.Series('', index=metadata.index)
    reason.loc[metadata.is_xenograft.str.lower().eq('true')] = 'xenograft'
    reason.loc[metadata.is_cell_line.str.lower().eq('true')] = 'cell_line'
    reason.loc[reason.eq('') & ~metadata.organism.eq('Homo sapiens')] = 'non_human'
    reason.loc[reason.eq('') & ~metadata.sample_type.eq('Tumor')] = 'not_patient_tumor'
    reason.loc[reason.eq('') & metadata.participant_id.isin(QUARANTINE)] = 'unresolved_participant_identifier'
    metadata['cohort_exclusion_reason'] = reason
    included_meta = metadata.loc[reason.eq('')]
    donors = sorted(included_meta.participant_id.unique())
    assert len(donors) == sum(PARTITION_COUNTS.values()) == 20
    ranked = sorted(donors, key=lambda d: hashlib.sha256(f'{SALT}|{d}'.encode()).hexdigest())
    assignment = {}
    cursor = 0
    for partition, count in PARTITION_COUNTS.items():
        assignment.update({d: partition for d in ranked[cursor:cursor + count]})
        cursor += count
    metadata['partition'] = metadata.participant_id.map(assignment).fillna('excluded')
    metadata.loc[reason.ne(''), 'partition'] = 'excluded'
    lib_reason = metadata.set_index('scpca_library_id').cohort_exclusion_reason
    obs['cohort_exclusion_reason'] = obs.library_id.map(lib_reason)
    obs['partition'] = obs.participant_id.map(assignment).fillna('excluded')
    obs.loc[obs.cohort_exclusion_reason.ne(''), 'partition'] = 'excluded'
    obs['technical_qc_pass'] = (
        obs.miQC_pass.eq(True) & obs.scDblFinder_class.eq('singlet') &
        obs.scpca_filter.eq('Keep') & obs.detected.ge(200) & obs['sum'].gt(0)
    )
    obs['analysis_eligible'] = obs.cohort_exclusion_reason.eq('') & obs.technical_qc_pass
    eligible = obs.loc[obs.analysis_eligible]
    assert eligible.groupby('participant_id').partition.nunique().max() == 1
    assert eligible.groupby('sample_id').partition.nunique().max() == 1
    assert eligible.groupby('library_id').partition.nunique().max() == 1
    assert not eligible.is_xenograft.any() and not eligible.is_cell_line.any()
    assert not eligible.participant_id.isin(QUARANTINE).any()
    assert set(eligible.participant_id) == set(assignment)

    donor_rows = []
    for donor in ranked:
        frame = eligible.loc[eligible.participant_id.eq(donor)]
        m = included_meta.loc[included_meta.participant_id.eq(donor)]
        donor_rows.append({
            'participant_id': donor, 'partition': assignment[donor],
            'split_rank': ranked.index(donor) + 1,
            'eligible_cells': len(frame), 'samples': frame.sample_id.nunique(),
            'libraries': frame.library_id.nunique(), 'capture_units': ';'.join(sorted(m.seq_unit.unique())),
        })
    donor_frame = pd.DataFrame(donor_rows)
    sample_rows = []
    for sample, m in included_meta.groupby('scpca_sample_id', sort=True):
        frame = eligible.loc[eligible.sample_id.eq(sample)]
        donor = m.participant_id.iloc[0]
        sample_rows.append({
            'sample_id': sample, 'participant_id': donor, 'partition': assignment[donor],
            'library_ids': ';'.join(sorted(m.scpca_library_id)),
            'capture_units': ';'.join(sorted(m.seq_unit.unique())),
            'eligible_cells': len(frame),
            'small_sample_flag': len(frame) < 100,
        })
    samples = pd.DataFrame(sample_rows)
    summary = []
    for partition in PARTITION_COUNTS:
        frame = eligible.loc[eligible.partition.eq(partition)]
        summary.append({'partition': partition, 'participants': frame.participant_id.nunique(),
                        'samples': frame.sample_id.nunique(), 'libraries': frame.library_id.nunique(),
                        'cells': len(frame)})
    assert {r['partition']: r['participants'] for r in summary} == PARTITION_COUNTS

    write_csv('donor_split.csv', donor_frame)
    write_csv('sample_manifest.csv', samples)
    write_csv('library_manifest.csv', metadata[['scpca_library_id', 'scpca_sample_id', 'participant_id',
              'sample_type', 'seq_unit', 'is_xenograft', 'is_cell_line', 'partition', 'cohort_exclusion_reason']])
    write_csv('partition_summary.csv', pd.DataFrame(summary))
    # Deterministic gzip metadata for reproducible manifest hashes.
    with (OUT / 'cell_manifest.csv.gz').open('wb') as raw:
        with gzip.GzipFile(fileobj=raw, mode='wb', mtime=0) as compressed:
            compressed.write(obs.rename(columns={'_index': 'obs_name'}).to_csv(index=False, lineterminator='\n').encode())

    exclusion_counts = obs.groupby('cohort_exclusion_reason', dropna=False).size().to_dict()
    source_stat = H5AD.stat()
    protocol = {
        'version': 'patient-benchmark-v1', 'frozen_utc': datetime.now(timezone.utc).isoformat(),
        'model_target': 'nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16',
        'split_status': 'Frozen donor assignment under released participant IDs; confirmatory interpretation subject to disclosed exploratory exposure',
        'split_method': 'Sort eligible participant IDs by SHA256(salt + | + participant_id); first 12 train, next 4 development, last 4 test. No label balancing, outcome inspection or split redraw.',
        'salt': SALT, 'assignment': assignment, 'partition_summary': summary,
        'cohort_rules': {'organism': 'Homo sapiens', 'sample_type': 'Tumor', 'is_xenograft': False, 'is_cell_line': False},
        'quarantined_participant_ids': sorted(QUARANTINE),
        'quarantine_rationale': 'These two IDs were flagged in the September source audit as potentially truncated. No aliases are inferred or merged. Their relationship to other IDs is unresolved; they remain outside all primary partitions.',
        'technical_qc': {'miQC_pass': True, 'scDblFinder_class': 'singlet', 'scpca_filter': 'Keep', 'detected_min': 200, 'sum_positive': True},
        'qc_note': 'scDblFinder filtering is a study choice, not an assertion that the portal removed predicted doublets. Excluded cells remain listed. Reference Unknown, disagreement and openscpca-excluded do not determine eligibility.',
        'total_source_cells': len(obs), 'eligible_cells': len(eligible),
        'cohort_cells_before_qc': int(obs.cohort_exclusion_reason.eq('').sum()),
        'cohort_exclusion_counts': exclusion_counts,
        'source_h5ad': {'path': str(H5AD), 'bytes': source_stat.st_size, 'sha256': sha256(H5AD), 'shape': shape, 'X_name': x_name},
        'source_metadata': {'path': str(META), 'sha256': sha256(META)},
        'source_obs_fields_read': read_fields,
        'source_code_sha256': sha256(ROOT/'notebooks/00_Data_Preparation.ipynb'),
        'environment': {'python': platform.python_version(), 'h5py': h5py.__version__, 'pandas': pd.__version__, 'numpy': np.__version__},
        'case_generation_contract': {
            'unit': 'Biological sample-specific cluster; documented same-sample technical libraries stay together.',
            'forbidden': ['Whole-cohort scVI embedding', 'Whole-cohort cluster labels/markers', 'Expert targets from the six whole-cohort pilot clusters', 'Case-specific held-out annotations in model prompts or retrieval'],
            'test_labels': 'No annotation columns were read for this freeze. Test case adjudication must be frozen before model comparison results are revealed.',
            'clustering': 'Not yet performed. Lock preprocessing, clustering and evidence serialization using training/development samples before generating final test cases.',
        },
        'exploratory_exposure': 'Investigators have already viewed whole-cohort embeddings, labels, 120B/4B predictions and six expert reviews spanning this dataset. These are held out from future fine-tuning, not previously unseen to the investigators. This is not prospective preregistration. Disclose this limitation; an independent external cohort would provide stronger confirmation.',
        'remaining_gates': ['Finalize count/log-normalization semantics against versioned conversion source before rebuilding expression features.', 'Freeze sample-specific clustering/evidence contract.', 'Create fresh sample-specific expert training/development/test panels.'],
    }
    protocol['output_sha256'] = {p.name: sha256(p) for p in sorted(OUT.iterdir()) if p.is_file()}
    (OUT / 'split_protocol.json').write_text(json.dumps(protocol, indent=2))
    report = ['# Patient benchmark v1', '', 'Donor assignments are frozen; case generation and expert test adjudication are pending.', '',
              '| Partition | Patients | Samples | Libraries | Eligible cells |', '|---|---:|---:|---:|---:|']
    for r in summary:
        report.append(f"| {r['partition']} | {r['participants']} | {r['samples']} | {r['libraries']} | {r['cells']:,} |")
    report += ['', f"Eligible: {len(eligible):,} cells, {eligible.participant_id.nunique()} released participant IDs. Before technical QC: {int(obs.cohort_exclusion_reason.eq('').sum()):,} primary-cohort cells.",
               '', 'Xenografts and cell lines are outside the primary cohort. SJNBL046 and SJNBL108 are quarantined pending donor-linkage clarification; no string-based alias merging was performed.',
               '', 'Technical QC requires portal Keep, miQC pass, predicted singlet, at least 200 detected genes and positive counts. Unknown annotations are retained. Small samples are flagged rather than silently removed.',
               '', 'Repeated specimens and technical libraries are assigned with their participant. Assignment uses only metadata and the fixed hash salt, without model outcomes or cell-type labels. Test labels were not read by this script.',
               '', 'The six reviewed whole-cohort clusters and their training drafts are pilot-only. Do not train on them in this patient-transfer experiment, because they may include future test donors.',
               '', 'Exploratory exposure is disclosed: whole-cohort data and outcomes were already inspected before this freeze. The test partition is held out from subsequent fine-tuning, not previously unseen to investigators. Do not describe this as a pristine prospective holdout.',
               '', 'Next: verify matrix semantics, lock per-sample preprocessing and clustering on training/development data, then build fresh expert panels. Do not reuse X_scVI or other merged all-patient embeddings.',
               '', 'Files: donor_split.csv, sample_manifest.csv, library_manifest.csv, cell_manifest.csv.gz, partition_summary.csv, split_protocol.json. Sources are opened read-only; original downloads are unchanged.']
    (OUT / 'README.md').write_text('\n'.join(report) + '\n')
    print(json.dumps({'output': str(OUT), 'partitions': summary, 'quarantined': sorted(QUARANTINE), 'eligible_cells': len(eligible)}, indent=2))


if __name__ == '__main__':
    main()

## 2. Training and validation cluster evidence
Normalize counts within each sample, cluster cells, and compute markers. The generator rejects test cells.

In [ ]:
import subprocess
subprocess.check_call([sys.executable, str(ROOT/'src/generate_sample_cases.py')])

## 3. Inspect technical QC
Inspect the per-sample plots and case manifest in `outputs/sample-cases-v1/`. Check small clusters, sample counts, and marker evidence before recording the recipe below. No test labels are used to select preprocessing settings.

## 4. Record the reviewed preprocessing recipe
Set `QC_REVIEWED = True` only after completing the preceding inspection. This preserves the configuration, source and environment checks needed for reserved-donor generation.

In [ ]:
"""Record a reviewed train/development recipe before generating reserved cases."""
import argparse,json
from pathlib import Path
import generate_sample_cases as recipe

def freeze(folder,qc_reviewed=False):
 if not qc_reviewed:raise ValueError('Review training/development technical QC first, then pass --qc-reviewed.')
 folder=Path(folder);manifest=json.loads((folder/'run_manifest.json').read_text())
 if manifest['test_cases_generated']!=0:raise ValueError('Cannot freeze a train/development recipe from test data.')
 if manifest['config']!=recipe.CONFIG:raise ValueError('Configuration differs from the generator.')
 if manifest['generator_sha256']!=recipe.digest(recipe.__file__):raise ValueError('Generator changed since case creation.')
 record={'status':'Train/development technical QC reviewed; recipe frozen before test generation','config':manifest['config'],'environment':manifest['environment'],'generator_sha256':manifest['generator_sha256']}
 target=folder/'case_recipe_v1.json';text=json.dumps(record,indent=2)+'\n'
 if target.exists()and json.loads(target.read_text())!=record:raise ValueError('A different frozen recipe already exists.')
 if not target.exists():target.write_text(text)
 return target

QC_REVIEWED = False
print(freeze(ROOT/"outputs/sample-cases-v1", qc_reviewed=QC_REVIEWED))

## 5. Reserved-donor cluster evidence
Run only after the recipe above is recorded. No expert reference annotations are read.

In [ ]:
subprocess.check_call([sys.executable, str(ROOT/'src/generate_reserved_test_cases.py')])

## 6. Test inputs and scoring protocol
Return here after expert test references and matched validation results are available locally. This locks the exact reference map, prompts, schemas, model/adapter and failure rules **before** test predictions. References stay local; the inference ZIP contains expression evidence and runtime modules only. Keep historical source hashes with their historical runs; this reorganized source produces a new run with new hashes.

Required inputs: `outputs/nemotron_4b_matched_development_v1.zip`, the expert test ledger under `outputs/test-review-67-cases/independent-expert-ledger/`, and the generated test evidence. The reviewer ledgers and inference archives are external inputs, not bundled data.

In [ ]:
import json, hashlib, shutil, zipfile
from collections import Counter
from datetime import datetime, timezone
from evaluation_metrics import REVIEW_SYSTEM, REVIEW_SCHEMA, DEVELOPMENT_LABEL_MAP
OUT = ROOT/'outputs/frozen-test-evaluation-v1'
PACKAGE = OUT/'inference-package'
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
OUT.mkdir(exist_ok=True);PACKAGE.mkdir(exist_ok=True)
with zipfile.ZipFile(ROOT/'outputs/nemotron_4b_matched_development_v1.zip') as z:
    original=json.loads(z.read('run_manifest.json'))
    sample=json.loads(z.read(next(n for n in z.namelist() if n.startswith('untuned/') and n.endswith('.json'))))
review_system=REVIEW_SYSTEM
review_schema=REVIEW_SCHEMA
refs_path=ROOT/'outputs/test-review-67-cases/independent-expert-ledger/test-independent-annotations.jsonl'
refs=[json.loads(s) for s in refs_path.read_text().splitlines()]
aliases=DEVELOPMENT_LABEL_MAP.copy()
categories={
    'neuronal_neuroendocrine':['Neuroblastic / sympathoadrenal','neuronal cells','neurons','neuroendocrine','neuroendocrine cells','neuroblastoma cells','neuroblasts','sympathoadrenal','sympathoadrenal cells','sympathetic neurons','chromaffin cells'],
    'myeloid':['Myeloid','Myeloid (macrophage-leaning)','Immune, myeloid-leaning','myeloid cells','macrophage','monocytes','monocyte','monocytes/macrophages','monocyte/macrophage','dendritic cells','neutrophils','granulocytes'],
    'schwann_glial':['Schwann-lineage','Schwann-lineage / Schwannian stroma','schwann','schwann cell','schwannian stroma','schwann-lineage cells','glial cells'],
    't_lymphoid':['T cells (cytotoxic)','t lymphocyte','cytotoxic t cells','cd4 t cells','cd8 t cells'],
    'stromal_mesenchymal':['Perivascular mural cells','Fibroblasts','Stromal (fibroblast/mural)','fibroblast','stromal','stromal cells','mesenchymal cells','mural cells','pericytes','pericyte','vascular smooth muscle cells','smooth muscle cells','perivascular cells'],
    'endothelial':['Endothelial','endothelial cell','vascular endothelial cells','blood vascular endothelial cells'],
    'b_lymphoid':['B cells','b cell','b lymphocytes','b lymphocyte'],
    'erythroid':['Erythroid','erythroid cells','erythrocytes','erythrocyte','red blood cells','red blood cell','rbc','rbcs'],
    'mixed_t_nk':['T / NK lymphocytes','t/nk lymphocytes','t/nk cells','t / nk cells','t and nk cells','t/nk','nk/t cells'],
}
normalize=lambda s:' '.join(s.casefold().split())
for group,names in categories.items():
    for label in names:aliases[normalize(label)]=group
references={r['case_id']:{'group':None if r['requires_further_review'] else aliases[normalize(r['broad_identity_verbatim'])],
                        'review':r['requires_further_review']} for r in refs}
scoring={
    'protocol':'reserved-donor-test-scoring-v1','locked_utc':datetime.now(timezone.utc).isoformat(),
    'status':'Locked before any test model predictions; expert references already observed by analyst.',
    'reference_ledger_sha256':sha(refs_path),'groups':list(categories),'label_map':aliases,
    'references':references,'source_aliases':'Frozen development aliases plus explicit pre-inference test-reference and biological synonyms.',
    'rules':[
        'All 67 cases remain in completion, review, acceptance, evidence and resource denominators.',
        'Identity agreement and proposed-label macro-F1 use the 57 references marked No further review.',
        'Yes (7) and Insufficient evidence (3) are review-positive references; original categories preserved.',
        'Use only exact label aliases after lowercasing and whitespace normalization. Unknown labels are unmapped and receive no lineage credit. No post-prediction map changes for primary scores.',
        'Macro-F1 equally weights the nine reference groups, including B lymphoid, erythroid and mixed T/NK. It is not directly comparable with six-class development macro-F1.',
        'Broad-lineage grouping collapses fibroblast and mural populations into stromal/mesenchymal as in development; subtype and malignancy are not validated.',
        'Mixed T/NK is distinct from specific T lymphoid, consistent with the prior conservative map.',
        'Only status complete is valid. Timeouts, invalid/partial JSON and missing answers count as failures and route operationally to review; do not count fallback as explicit uncertainty recognition.',
        'Compare original identity decisions and, separately, the fixed review-first gate applied to those same identity outputs; do not regenerate gated identities.',
        'Report accepted resolved agreement separately from supported correct yield among all accepted cases; keep expert-uncertain acceptances explicit.',
        'Report donor-level counts for four donors, not patient-level inference from cluster counts.',
        'Supporting-gene presence/zero/duplicates are traceability checks, not biological reasoning scores.',
        'Freeze V1, both prompts, schemas, limits and failure handling. No selection or tuning from test outcomes.',
    ],
    'identity_reference_group_counts':dict(Counter(r['group'] for r in references.values() if not r['review'])),
    'donors':{r['case_id']:r['participant_id'] for r in refs},
}
(OUT/'scoring-protocol.json').write_text(json.dumps(scoring,indent=2))
for name in ['matched_evaluation.py','frozen_test_evaluation.py']:shutil.copyfile(ROOT/'src'/name,PACKAGE/name)
shutil.copyfile(ROOT/'outputs/reserved-test-cases-v1/case-evidence.jsonl',PACKAGE/'case-evidence.jsonl')
config={
    'protocol':'frozen-reserved-donor-test-v1','model_id':original['model_id'],'revision':original['revision'],
    'adapter_hashes':original['adapter_hashes'],'software':original['software'],'gpu':original['gpu'],
    'case_ids':sorted(references),'identity_system':sample['messages'][0]['content'],'review_system':review_system,
    'files':{name:sha(PACKAGE/name) for name in ['matched_evaluation.py','frozen_test_evaluation.py','case-evidence.jsonl']},
    'scoring_lock_sha256':sha(OUT/'scoring-protocol.json'),'scoring_reference_sha256':sha(refs_path),
    'stages':{
        'identity':{'schema':original['schema'],'decoding':original['decoding']},
        'review_first':{'schema':review_schema,'decoding':{**original['decoding'],'max_new_tokens':256,'max_time_seconds':45}},
    },
}
(PACKAGE/'config.json').write_text(json.dumps(config,indent=2))
archive=OUT/'frozen-test-inference-v1.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['config.json','case-evidence.jsonl','matched_evaluation.py','frozen_test_evaluation.py']:
        z.write(PACKAGE/name,name)

print('Inference ZIP:', archive)
print('EXPECTED_PACKAGE_SHA256:', sha(archive))